In [25]:
# Paste in Jupyter, paste output back
from pymongo import MongoClient
import os

uri = os.environ.get("MONGO_URI", "mongodb://localhost:27017")
client = MongoClient(uri)
db = client[os.environ.get("MONGO_DATABASE", "quants_lab")]

pipeline = [
    {"$match": {"trading_pair": "XMR-USDT", "connector": {"$in": ["mexc", "nonkyc"]}}},
    {"$group": {
        "_id": {"connector": "$connector", "interval": "$interval"},
        "count": {"$sum": 1},
        "first_ts": {"$min": "$timestamp"},
        "last_ts":  {"$max": "$timestamp"},
    }},
    {"$sort": {"_id.connector": 1, "_id.interval": 1}},
]
for doc in db["candles"].aggregate(pipeline):
    c = doc["_id"]["connector"]
    i = doc["_id"]["interval"]
    n = doc["count"]
    span_days = (doc["last_ts"] - doc["first_ts"]) / 86400
    print(f"{c:>8}  {i:>4}  {n:>7} bars   {span_days:6.1f} days")

    mexc   15m    34599 bars    360.4 days
    mexc    1d     2318 bars   2317.0 days
    mexc    1h     5410 bars    225.4 days
    mexc    1m   108735 bars     75.5 days
    mexc    4h     1351 bars    225.0 days
    mexc    5m   103798 bars    360.4 days
    mexc    8h      540 bars    179.7 days
  nonkyc   12h      360 bars    179.5 days
  nonkyc   15m    96889 bars   1053.5 days
  nonkyc    1d     1079 bars   1079.0 days
  nonkyc    1h    25541 bars   1079.5 days
  nonkyc    4h     6446 bars   1079.3 days
  nonkyc    5m   201274 bars    730.5 days
  nonkyc    8h      540 bars    179.7 days


In [26]:
# verify_quants_lab_probes.py
# Run this in the pmm_dynamic directory (/mnt/.../market_lab/pmm_dynamic)

import sys, os, json, hashlib, inspect
from pathlib import Path

# 1. Confirm MongoCandleLoader methods at runtime
from pmm_lab.data.mongo import MongoCandleLoader
print("MongoCandleLoader methods:")
print(sorted([m for m in dir(MongoCandleLoader) if not m.startswith('_')]))
print("has .load:", hasattr(MongoCandleLoader, "load"))
print("has .load_range:", hasattr(MongoCandleLoader, "load_range"))

# 2. Inspect run_simulation dispatch — does it accept directional configs?
from pmm_lab.sim.runner_dispatch import run_simulation
print("\nrun_simulation signature:")
print(inspect.signature(run_simulation))
src = inspect.getsource(run_simulation)
print("\nrun_simulation source (first 2000 chars):")
print(src[:2000])

# 3. Check hash_candles return type
from pmm_lab.data.hashing import hash_candles
import numpy as np
sample = np.zeros(10, dtype=[("timestamp", "i8"), ("open", "f8"), ("high", "f8"),
                              ("low", "f8"), ("close", "f8"), ("volume", "f8")])
h = hash_candles(sample)
print(f"\nhash_candles return type: {type(h).__name__}, value: {h!r}")

# 4. Confirm directional modules import without pandas_ta in current env
try:
    import pandas_ta
    print("\npandas_ta is installed, version:", pandas_ta.__version__)
except ImportError:
    print("\npandas_ta NOT installed")

# Does importing directional-only modules pull in pandas_ta?
try:
    # Fresh import check
    import importlib
    for mod in list(sys.modules):
        if mod.startswith("pmm_lab.strategies") or mod.startswith("pmm_lab.features"):
            del sys.modules[mod]
    from pmm_lab.strategies.mean_reversion_bb_rsi import MeanReversionBBRSIStrategyConfig
    print("MR import: OK")
    from pmm_lab.strategies.ema_regime_hold import EMARegimeHoldStrategyConfig
    print("EMA import: OK")
except Exception as e:
    print(f"Directional import failed: {type(e).__name__}: {e}")

# 5. Check strategies/__init__.py content
init_path = Path("pmm_lab/strategies/__init__.py")
print(f"\npmm_lab/strategies/__init__.py content:")
print(init_path.read_text())

# 6. Check runner_dispatch for except ImportError patterns
dispatch_path = Path("pmm_lab/sim/runner_dispatch.py")
src = dispatch_path.read_text()
print(f"\nrunner_dispatch.py has {src.count('except ImportError')} `except ImportError` blocks")
print(f"runner_dispatch.py total lines: {len(src.splitlines())}")

# 7. Confirm result_entry schema doesn't have binding_frac
cell8 = Path("notebooks/direction-custom/_build_cell8.py").read_text()
print(f"\n_build_cell8.py:")
print(f"  'binding_frac' occurrences: {cell8.count('binding_frac')}")
print(f"  'max_trades_per_day_binding_fraction' occurrences: {cell8.count('max_trades_per_day_binding_fraction')}")

MongoCandleLoader methods:
['ensure_indexes', 'last_raw_duplicate_count', 'list_combos', 'load_range', 'ping']
has .load: False
has .load_range: True

run_simulation signature:
(config: 'Any', pair_rules: 'PairRules', candles: 'np.ndarray', precomputed_signals, engine_config=None, sim_start_idx: 'Optional[int]' = None, bar_index_offset: 'int' = 0, regime_candles: 'Optional[np.ndarray]' = None) -> 'SimResult'

run_simulation source (first 2000 chars):
def run_simulation(
    config: Any,
    pair_rules: PairRules,
    candles: np.ndarray,
    precomputed_signals,
    engine_config=None,
    sim_start_idx: Optional[int] = None,
    bar_index_offset: int = 0,
    regime_candles: Optional[np.ndarray] = None,
) -> SimResult:
    """Run a backtest with precomputed signals for any supported config type.

    Parameters
    ----------
    config : SimConfig | MeanReversionBBRSIStrategyConfig | EMARegimeHoldStrategyConfig
        Strategy configuration.
    pair_rules : PairRules
        Exchan

AttributeError: module 'pandas_ta' has no attribute '__version__'